In [5]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Load and preprocess data
df = pd.read_csv('../data/bitcoin_timeseries.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')
df_hourly = df.resample('1h').mean().dropna()
train = df_hourly[:-24]
test = df_hourly[-24:]

In [7]:
# Fit ARIMA model with fixed order
model_arima = ARIMA(train, order=(2,1,2))
fitted_arima = model_arima.fit()
forecast_arima = fitted_arima.forecast(steps=len(test))

In [8]:
# Fit SARIMA model with fixed seasonal order
model_sarima = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,24))
fitted_sarima = model_sarima.fit()
forecast_sarima = fitted_sarima.forecast(steps=len(test))



KeyboardInterrupt: 

In [ ]:
# Generate forecast timestamps
last_timestamp = train.index[-1]
forecast_index = [last_timestamp + timedelta(hours=i) for i in range(1, len(test) + 1)]

# Save forecasts
df_comparison = pd.DataFrame({
    'timestamp': forecast_index,
    'arima_pred': forecast_arima.values,
    'sarima_pred': forecast_sarima.values
})
df_comparison.to_csv('../reports/arima_vs_sarima_forecast.csv', index=False)

In [ ]:
# Plot
plt.figure(figsize=(12, 5))
plt.plot(forecast_index, forecast_arima, label='ARIMA', marker='o')
plt.plot(forecast_index, forecast_sarima, label='SARIMA', marker='x')
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.xlabel('Timestamp')
plt.ylabel('BTC Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Save metrics
mae_arima = mean_absolute_error(test, forecast_arima)
rmse_arima = mean_squared_error(test, forecast_arima, squared=False)
mae_sarima = mean_absolute_error(test, forecast_sarima)
rmse_sarima = mean_squared_error(test, forecast_sarima, squared=False)

with open('../reports/arima_vs_sarima_metrics.txt', 'w') as f:
    f.write(f'ARIMA MAE: {mae_arima:.2f}, RMSE: {rmse_arima:.2f}\n')
    f.write(f'SARIMA MAE: {mae_sarima:.2f}, RMSE: {rmse_sarima:.2f}\n')